In [1]:
import pandas as pd

df = pd.read_csv("buzzword_dilution_dataset.csv",encoding="latin1")

texts = df["text"].astype(str).tolist()
years = df["year"].tolist()
labels = df["label"].tolist()
buzzwords = df["buzzword"].tolist()


In [2]:
from docx import Document
import re

def load_official_doc(path):
    doc = Document(path)
    text = " ".join(p.text for p in doc.paragraphs if p.text.strip())
    text = re.sub(r"\s+", " ", text)

    sentences = text.split(".")
    chunks = [
        ". ".join(sentences[i:i+3]).strip()
        for i in range(0, len(sentences), 3)
        if sentences[i].strip()
    ]
    return chunks

official_chunks = load_official_doc(
    "Buzzwords official document.docx"
)


In [3]:
import torch
import torch.nn as nn
from transformers import BertModel

class SentenceTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")

    def forward(self, input_ids, token_type_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            token_type_ids=token_type_ids,
            attention_mask=attention_mask
        )

        # outputs.last_hidden_state → (batch, seq_len, 768)
        token_embeddings = outputs.last_hidden_state

        # Mean pooling
        attention_mask = attention_mask.unsqueeze(-1)
        sentence_embedding = torch.sum(
            token_embeddings * attention_mask, dim=1
        ) / torch.clamp(attention_mask.sum(dim=1), min=1e-9)

        return sentence_embedding


c:\Sakshi(MSC DS)\Sem 4\Research Paper\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = SentenceTransformer()
model.eval()

def encode_sentences(sentences):
    embeddings = []

    for sent in sentences:
        tokens = tokenizer(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True
        )

        with torch.no_grad():
            emb = model(
                tokens["input_ids"],
                tokens["token_type_ids"],
                tokens["attention_mask"]
            )

        embeddings.append(emb.squeeze(0))

    return torch.stack(embeddings)


In [5]:
official_embeddings = encode_sentences(official_chunks)

dataset_embeddings = encode_sentences(texts)


In [6]:
from torch.nn.functional import cosine_similarity

def compute_max_similarity(dataset_embs, official_embs):
    similarities = []

    for emb in dataset_embs:
        sim = cosine_similarity(
            emb.unsqueeze(0),
            official_embs
        )
        similarities.append(sim.max().item())

    return similarities


In [7]:
similarities = compute_max_similarity(dataset_embeddings, official_embeddings)

print(similarities[:10])


[0.854580283164978, 0.8643947839736938, 0.9115368127822876, 0.9013687372207642, 0.856613278388977, 0.8998788595199585, 0.9231077432632446, 0.8743858933448792, 0.8679088950157166, 0.9526432752609253]
